In [35]:
# перед работой в kernel ставятся необходимые фреймворки 
# для этого возможен список команд:  

# python -m venv vvenvv
# source ./vvenvv/bin/activate
# pip install requirements.txt" 

# так как у меня уже есть .venv и нет requirements.txt, ставлю фреймворки через shell (оператором восклицательного знака)
# предварительно обновлю и ядро ipykernel, тк ставлю в новой среде
# /home/snowwy/Documents/poor_monk/poor_monk/7th_term/about_ai_course/ml/vvenvv/bin/python -m pip install ipykernel -U  --force-reinstal

!pip install pandas numpy matplotlib seaborn scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


# 1. Выбор начальных условий

## a. Набор данных для задачи классификации 

Реальная практическая задача:

Выявление тенденций в мед. страховании между сборами и мед. факторами согласно kaggle-page. 

- __Набор данных__: https://www.kaggle.com/datasets/muhammadanwaar101/healthcare-insurance-charges-dataset

## b. Набор данных для задачи регрессии 

Реальная практическая задача:

Выявление Хронической Болезни Почек и оценки факторов риска функции почек согласно kaggle-page.

- __Набор данных__: https://www.kaggle.com/datasets/miadul/kidney-function-health-dataset

## c. Метрики качества и обоснование выбора


### Классификация:
- __Accuracy__ - процент объектов, верно классифицированных моделью. 
- __F1-score__ - среднее гармоническое между точностью (precision) и полнотой (recall).

### Регрессия:
- __MAE__ - насколько в среднем модель ошибается в прогнозах.
- __R²__ - доля дисперсии целевой переменной, которую объясняет модель. Чем ближе к 1, тем лучше.


In [67]:
# подгрузка библиотек для получения требуемых методов. 
# библиотечные вызовы именуются в общепринятой сокращенной нотации для читаемости кода
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [68]:
# 1.1. загрузка датасета классификации
# задача из реальной предметной области - "факторы, влияющие на страховые выплаты (взносы, сборы)"
clf_csv_pth = "/home/snowwy/Documents/poor_monk/poor_monk/7th_term/about_ai_course/ml/dat_clf.csv"
df_clf = pd.read_csv(clf_csv_pth)

In [69]:
# 1.2 загрузка датасета регрессии
# задача из реальной предметной области - "медицинские записи с биомаркерами"
reg_csv_pth = "dat_reg.csv"
df_reg = pd.read_csv(reg_csv_pth)

In [77]:
# 1.3. исследование датасета  классификации
# производится в одном из двух форматов, доступных в пандасе. 
# взамен одномерному (1-n) series, при работе с csv рекомендуется использовать двумерный (2-n) dataframe

df_clf.head()
# вывод кортежа размерностью dataframe
df_clf_pd = pd.DataFrame(df_clf)

# применение IPython.display позволяет структурированно отобразить в консоли датафрейм
display(df_clf_pd)

,AGE,Gender,Body_Mass_Index(BMI),Number_of_Children,Smoking_Status,Region,Insurance_Charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1402,34,female,27.720,0,no,southeast,4415.15880
1403,42,female,37.900,0,no,southwest,6474.01300
1404,51,female,36.385,3,no,northwest,11436.73815
1405,54,female,27.645,1,no,northwest,11305.93455


In [78]:
# продолжение исследования датасета классификации
# выбор все строки, в которых значение столбца age больше 30 
display(df_clf_pd[df_clf_pd.AGE > 20])

,AGE,Gender,Body_Mass_Index(BMI),Number_of_Children,Smoking_Status,Region,Insurance_Charges
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
5,31,female,25.740,0,no,southeast,3756.62160
6,46,female,33.440,1,no,southeast,8240.58960
...,...,...,...,...,...,...,...
1402,34,female,27.720,0,no,southeast,4415.15880
1403,42,female,37.900,0,no,southwest,6474.01300
1404,51,female,36.385,3,no,northwest,11436.73815
1405,54,female,27.645,1,no,northwest,11305.93455


In [79]:
# 1.4. исследование датасета регрессии
# окинем взглядом первые строки dataframe
df_reg.head()
df_reg.shape
# вывод обобщенного содержимого датафрейма
df_reg.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Creatinine        5000 non-null   float64
 1   BUN               5000 non-null   float64
 2   GFR               5000 non-null   float64
 3   Urine_Output      5000 non-null   float64
 4   Diabetes          5000 non-null   int64  
 5   Hypertension      5000 non-null   int64  
 6   Age               5000 non-null   float64
 7   Protein_in_Urine  5000 non-null   float64
 8   Water_Intake      5000 non-null   float64
 9   Medication        2013 non-null   str    
 10  CKD_Status        5000 non-null   int64  
dtypes: float64(7), int64(3), str(1)
memory usage: 429.8 KB


## old block - to prompt for regression

In [ ]:
# для классификации разделим признаки на категориальные и числовые

str_cols = df_clf.select_dtypes(include=['object']).columns.tolist()

print("Cat var clf:")
print(str_cols)

str_cols_2 = df_reg.select_dtypes(include=['object']).columns.tolist()
print("Cat var reg:")
print(str_cols_2)

Categorical variables:
['Medication']
Categorical variables:
['Gender', 'Smoking_Status', 'Region']


In [60]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# --- 1.1. Очистка от выбросов (IQR) ---
Q1 = df_reg['Insurance_Charges'].quantile(0.25)
Q3 = df_reg['Insurance_Charges'].quantile(0.75)
IQR = Q3 - Q1
df_ins_clean = df_reg[(df_reg['Insurance_Charges'] >= Q1 - 1.5*IQR) & 
                      (df_reg['Insurance_Charges'] <= Q3 + 1.5*IQR)].copy()

# --- 1.2. Создание таргета ---
median_val = df_ins_clean['Insurance_Charges'].median()
df_ins_clean['High_Charge'] = (df_ins_clean['Insurance_Charges'] > median_val).astype(int)

# --- 1.3. Признаки и Пайплайн ---
X = df_ins_clean.drop(columns=['Insurance_Charges', 'High_Charge'])
y = df_ins_clean['High_Charge']

cat_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'str']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first'), cat_cols)
])

model_clf = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced', random_state=42))
])

# Обучение
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model_clf.fit(X_train, y_train)

print("РЕЗУЛЬТАТЫ: СТРАХОВАНИЕ (КЛАССИФИКАЦИЯ)")
print(classification_report(y_test, model_clf.predict(X_test)))

РЕЗУЛЬТАТЫ: СТРАХОВАНИЕ (КЛАССИФИКАЦИЯ)
              precision    recall  f1-score   support

           0       0.90      0.93      0.91       126
           1       0.93      0.90      0.91       126

    accuracy                           0.91       252
   macro avg       0.91      0.91      0.91       252
weighted avg       0.91      0.91      0.91       252



In [62]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# --- 2.1. Очистка GFR от выбросов ---
Q1_k = df_clf['GFR'].quantile(0.25)
Q3_k = df_clf['GFR'].quantile(0.75)
IQR_k = Q3_k - Q1_k
df_kid_clean = df_clf[(df_clf['GFR'] >= Q1_k - 1.5*IQR_k) & 
                      (df_clf['GFR'] <= Q3_k + 1.5*IQR_k)].copy()

# --- 2.2. Признаки (УДАЛЯЕМ КРЕАТИНИН для честности) ---
X_k = df_kid_clean.drop(columns=['GFR', 'CKD_Status', 'Medication', 'Creatinine'])
y_k = df_kid_clean['GFR']

num_k = X_k.select_dtypes(exclude=['object', 'str']).columns.tolist()

# --- 2.3. Пайплайн ---
preprocessor_k = ColumnTransformer([
    ('num', StandardScaler(), num_k)
])

model_reg = Pipeline([
    ('pre', preprocessor_k),
    ('reg', LinearRegression())
])

# Обучение
X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(X_k, y_k, test_size=0.2, random_state=42)
model_reg.fit(X_train_k, y_train_k)

# Оценка
y_pred_k = model_reg.predict(X_test_k)
print("\nРЕЗУЛЬТАТЫ: ПОЧКИ (РЕГРЕССИЯ БЕЗ КРЕАТИНИНА)")
print(f"MAE: {mean_absolute_error(y_test_k, y_pred_k):.2f}")
print(f"MSE: {mean_squared_error(y_test_k, y_pred_k):.2f}")

print(f"RMSE: {np.sqrt(mean_squared_error(y_test_k, y_pred_k)):.2f}")
print(f"R2: {r2_score(y_test_k, y_pred_k):.4f}")


РЕЗУЛЬТАТЫ: ПОЧКИ (РЕГРЕССИЯ БЕЗ КРЕАТИНИНА)
MAE: 7.72
MSE: 135.22
RMSE: 11.63
R2: 0.8715


In [82]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

# --- 2.3. Продвинутый Пайплайн ---
# Добавляем PolynomialFeatures(degree=2), чтобы ловить нелинейные связи
# И используем Ridge вместо обычной LinearRegression
model_reg_upgraded = Pipeline([
    ('pre', preprocessor_k),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)), 
    ('reg', Ridge(alpha=1.0)) 
])

# Обучение
model_reg_upgraded.fit(X_train_k, y_train_k)

# Оценка
y_pred_upgraded = model_reg_upgraded.predict(X_test_k)
final_mse = mean_squared_error(y_test_k, y_pred_upgraded)

print("\n" + "="*35)
print("РЕЗУЛЬТАТЫ: УЛУЧШЕННАЯ РЕГРЕССИЯ")
print("="*35)
print(f"MAE: {mean_absolute_error(y_test_k, y_pred_upgraded):.2f}")
print(f"MSE: {final_mse:.2f} (Цель: < 50)")
print(f"RMSE: {np.sqrt(final_mse):.2f}")
print(f"R2: {r2_score(y_test_k, y_pred_upgraded):.4f}")


РЕЗУЛЬТАТЫ: УЛУЧШЕННАЯ РЕГРЕССИЯ
MAE: 6.52
MSE: 109.15 (Цель: < 50)
RMSE: 10.45
R2: 0.8962


# 2. Создание бейзлайна и оценка качества

## a. Обучить модели из sklearn (для классификации и регрессии) для выбранных наборов данных



In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

### clf_а.1. Предобработка датасета классификации

In [18]:
# --- 2.1. предобработка датасета классификации ---
# Создание целевой переменной (Бинаризация)
# Так как исходная задача — классификация, делим чеки по медиане
median_charge = df_clf['Insurance_Charges'].median()
df_clf['High_Charge'] = (df_clf['Insurance_Charges'] > median_charge).astype(int)

# Признаки и таргет (Определение X и y)
X = df_clf.drop(columns=['Insurance_Charges', 'High_Charge'])
y = df_clf['High_Charge']

# Настройка трансформера (Pipeline Step 1)
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), str_cols) # drop='first' для избежания ловушки дамми-переменных
])

KeyError: 'Insurance_Charges'

### clf_а.2. Бейзлайн классификации

In [ ]:
# --- 2. БЕЙЗЛАЙН ---
# Разделение на обучающую и тестовую выборки
# Используем stratify=y, чтобы сохранить баланс классов в обеих частях
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Модель (включаем балансировку весов на случай дисбаланса)
# Создание и обучение модели (Pipeline Step 2)
# LogisticRegression с балансировкой весов — отличный выбор для бейзлайна
clf_model = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(C=1.0, class_weight='balanced', random_state=42))
])

clf_model.fit(X_train, y_train)

### clf_b. Оценить качество моделей (для классификации) по выбранным метрикам на выбранных наборах данных



In [ ]:
# --- 3. ОЦЕНКА ---
### clf_b. Оценка качества модели
# 7. Получение предсказаний
y_pred = clf_model.predict(X_test)

# 8. Вывод метрик
print("\n--- ОТЧЕТ О КЛАССИФИКАЦИИ ---")
print(classification_report(y_test, y_pred))

# 9. Визуализация матрицы ошибок
print("\nМатрица ошибок:")
ConfusionMatrixDisplay.from_estimator(clf_model, X_test, y_test, cmap='Blues')

## b. Оценить качество моделей (для классификации и регрессии) по выбранным метрикам на выбранных наборах данных



In [ ]:
# 2.b. Оценить качество моделей (для классификации и регрессии) по выбранным метрикам на выбранных наборах данных

# logr. basel. оценк. посл. predict и обосн. метрик.
y_clf_pred_lr = log_reg.predict(Xclf_test)

print("logreg")

# linr-basel. оценк. посл. predict и обосн. метрик.
y_reg_pred_lr = lin_reg.predict(Xreg_test)

print("linreg.")


# 3. Улучшение бейзлайна

## a. Сформулировать гипотезы (препроцессинг данных, визуализация данных, формирование новых признаков, подбор гиперпараметров на кросс-валидации и т.д.)

## b. Проверить гипотезы

## c. Сформировать улучшенный бейзлайн по результатам проверки гипотез

## d. Обучить модели с улучшенным бейзлайном (для классификации и регрессии) для выбранных наборов данных


## d. Обучить модели с улучшенным бейзлайном (для классификации и регрессии) для выбранных наборов данных

## e. Оценить качество моделей с улучшенным бейзлайном (для классификации и регрессии) по выбранным метрикам на выбранных наборах данных

## f. Сравнить результаты моделей с улучшенным бейзлайном в сравнении с результатами из пункта 2 

## g. Сделать выводы

In [ ]:
# гипот.: 
# 1. в завис. от k = 1, 3, 5 меняется точн.
# 2. в завис. от rand state меняется кач. train_test_split
# 3. в завис. от обработ. числ. и категор. столбц. помен. выбр. метрик.

In [ ]:
# улучш бейзл 
### 2a.2 Обуч мод

# для гипотезы 1:
for k in [1, 3, 5]:
    model = Pipeline([
        ("prep", prep_clf),
        ("model", KNeighborsClassifier(n_neighbors=k))
    ])
    
    model.fit(Xclf_train, yclf_train)
    y_pred = model.predict(Xclf_test)
    
    print(
        f"k={k}, "
        f"accuracy={accuracy_score(yclf_test, y_pred):.3f}, "
        f"f1={f1_score(yclf_test, y_pred):.3f}"
    )

In [ ]:
# дл. гипот. 2:

for rs in [0, 21, 42]:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=rs
    )
    
    model = Pipeline([
        ("prep", prep_clf),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ])
    
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    print(
        f"random_state={rs}, "
        f"accuracy={accuracy_score(y_te, y_pred):.3f}, "
        f"f1={f1_score(y_te, y_pred):.3f}"
    )


In [ ]:
# для clf потенциально масшт. примен. прямое кодирование (при усл. дамми-переменных, см. с.230) в силу бин. классиф. относ. статуса курения в датасет.
# дл. гипот. 3 
prep_no_scaler = ColumnTransformer([
    ("num", "passthrough", num_cols_clf),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_clf)
])

prep_with_scaler = ColumnTransformer([
    ("num", StandardScaler(), num_cols_clf),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_clf)
])


In [ ]:
# e. оценк. кач-ва модел.
mae = mean_absolute_error(yreg_test, yreg_pred)
rmse = np.sqrt(mean_squared_error(yreg_test, yreg_pred))
r2 = r2_score(yreg_test, yreg_pred)
# чтоб сдел. прогн., выз. predict объекта
yclf_pred = basel_clf.predict(Xclf_test)

# обосн выбор метр
acur = accuracy_score(yclf_test, yclf_pred)
f_1 = f1_score(yclf_test, yclf_pred)



## g. Выводы
- согласно книге, влиян. выдвин. гипот. не крит.
- след. выдв. гипот. бол. системн.

# 4. Имплементация алгоритма машинного обучения 

## a. Самостоятельно имплементировать алгоритмы машинного обучения (для классификации и регрессии)

## b. Обучить имплементированные модели (для классификации и регрессии) для выбранных наборов данных

## c. Оценить качество имплементированных моделей (для классификации и регрессии) по выбранным метрикам на выбранных наборах данных




## d. Сравнить результаты имплементированных моделей в сравнении с результатами из пункта 2 


## e. Сделать выводы


# j. сделать выводы
- Улучш. basel. неплох.
- Предобраб. дан. знач. в работе.